In [1]:
!pip install -U langchain-huggingface sentence-transformers
!pip install -U langchain
!pip install -U langchain-experimental


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 23.7 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 26.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.18
    Uninstalling langchain-1.3.18:
      Successfully uninstalled langchain-1.3.18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import json
from pathlib import Path

# HuggingFaceEmbeddings for generating embeddings...
from langchain_huggingface import HuggingFaceEmbeddings

# SemanticChunker for similarity check... 
from langchain_experimental.text_splitter import SemanticChunker


# loading embeddings model...
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# giving embeddings to SemanticChunker for Similarity check...
chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile"
)

# specifying the pdfs data directory...
input_dir = Path("/content/drive/MyDrive/DS RAG-PROJECT/cleaned_data")
output_dir = Path("/content/drive/MyDrive/DS RAG-PROJECT/chunks_data")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# reading each file in directory...
for file in input_dir.glob("*.json"):

  chunks = []

  # open each file...
  with open(file, 'r', encoding='utf-8') as f:
    context = json.load(f)

  # read each page in each file...
  for page in context:

    text = page["text"].strip()  # extract text for embeddings...
    if not text:
      continue

    # Give the text to SemanticChunker.
    # It creates embeddings for the sentences, compares their semantic similarity,
    # and then groups semantically similar sentences into chunks.
    text_splitter = chunker.split_text(text)  

    # extracting metadata...
    metadata = {
        key: value
        for key, value in page.items()
        if key != "text"
    }

    # Loop through the chunks of each page and file,
    # and store each chunk along with its metadata in the chunks list.
    for chunk in text_splitter:
      chunks.append({
          "text": chunk,
          "metadata": metadata
      })

  # specifying output path and files names...
  output_path = output_dir / f"{file.stem}_chunks.json"

  # encoding='utf-8' → supports different characters and languages
  # ensure_ascii=False → keeps non-ASCII characters readable
  # indent=4 → formats the JSON with 4-space indentation
  with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=4)

  print(
      f"Processed: {file.stem} | "
      f"Chunks: {len(chunks)}"
  )
print("\nAll files processed! ✌")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Processed: Data_Collection_clean | Chunks: 23
Processed: Data_visualization_clean | Chunks: 45
Processed: Flask_clean | Chunks: 42
Processed: Git_&_GitHub_clean | Chunks: 56
Processed: Introduction_to_Data_Science_clean | Chunks: 24
Processed: Introduction_to_Google_Colab_clean | Chunks: 15
Processed: LLMs_clean | Chunks: 22
Processed: Machine_Learning_clean | Chunks: 23
Processed: ML_Algorithms_clean | Chunks: 30
Processed: Neural_Networks_clean | Chunks: 41
Processed: Numpy_clean | Chunks: 45
Processed: Pandas_clean | Chunks: 58
Processed: Probability_clean | Chunks: 33
Processed: Probability_Distributions_clean | Chunks: 25
Processed: Probability_Study_Guide_clean | Chunks: 72
Processed: Python_Refresher_clean | Chunks: 89
Processed: Scikit_Learn_clean | Chunks: 76
Processed: SQL_clean | Chunks: 125
Processed: Understanding_the_Conda_Environment_clean | Chunks: 16

All files processed! ✌
